In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/jigyasadoharey/tfidf-model/__results__.html
/kaggle/input/notebooks/jigyasadoharey/tfidf-model/submission.csv
/kaggle/input/notebooks/jigyasadoharey/tfidf-model/__notebook__.ipynb
/kaggle/input/notebooks/jigyasadoharey/tfidf-model/__output__.json
/kaggle/input/notebooks/jigyasadoharey/tfidf-model/tfidf.pkl
/kaggle/input/notebooks/jigyasadoharey/tfidf-model/custom.css
/kaggle/input/notebooks/jigyasadoharey/tfidf-model/wandb/run-20260625_183253-pjniy5yp/run-pjniy5yp.wandb
/kaggle/input/notebooks/jigyasadoharey/tfidf-model/wandb/run-20260625_183253-pjniy5yp/logs/debug.log
/kaggle/input/notebooks/jigyasadoharey/tfidf-model/wandb/run-20260625_183253-pjniy5yp/logs/debug-internal.log
/kaggle/input/notebooks/jigyasadoharey/tfidf-model/wandb/run-20260625_183253-pjniy5yp/files/wandb-summary.json
/kaggle/input/notebooks/jigyasadoharey/tfidf-model/wandb/run-20260625_183253-pjniy5yp/files/config.yaml
/kaggle/input/notebooks/jigyasadoharey/tfidf-model/wandb/run-20260625_18325

In [2]:
import joblib
bundle = joblib.load("/kaggle/input/notebooks/jigyasadoharey/tfidf-model/tfidf.pkl")

In [3]:
test = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

In [4]:
test_rows = []

options = ['A', 'B', 'C', 'D', 'E']

for _, row in test.iterrows():

    prompt = row['prompt']

    for opt in options:

        text = prompt + ' [SEP] ' + str(row[opt])

        test_rows.append({
            'id': row['id'],
            'text': text,
            'option': opt
        })

new_test = pd.DataFrame(test_rows)

In [5]:
vectorizer = bundle['vectorizer']
model = bundle['model']

In [6]:
X_test_tfidf = vectorizer.transform(new_test['text'])

probs = model.predict_proba(X_test_tfidf)[:,1]

new_test['prob'] = probs

In [7]:
final_predictions = []

for qid in new_test['id'].unique():

    temp = new_test[new_test['id'] == qid]

    temp = temp.sort_values(by='prob', ascending=False)

    top3 = temp['option'].values[:3]

    prediction = ' '.join(top3)

    final_predictions.append({
        'id': qid,
        'prediction': prediction
    })

submission = pd.DataFrame(final_predictions)

print(submission.head())

submission.to_csv('submission.csv', index=False)

   id prediction
0   1      A E C
1   2      B E A
2   3      B E D
3   4      E C A
4   5      C A D
